# 18 Sequence Packing 如何减少 padding，又如何保持样本独立？

## 面试回答主线

Sequence packing 把多个短样本拼进固定长度 block，提高 token 利用率；关键是同时维护 position、segment/document id、loss mask 与 attention mask，不能把简单 concat 当作完成。面试中需区分“只是把 token 放在一起”和“模型不会跨样本注意”的差别。实验把六段客服对话 token 按 10 token block 贪心拼接，比较普通 padding 槽位、packing 利用率和 segment id；随后展示漏 segment mask 后跨文档 attention 泄漏。

**核心公式：** packing 后 mask 应满足 $M_{ij}=1$ 当且仅当 $j\le i$ 且 $s_i=s_j$；position 可在每段重置或连续，必须与训练/推理约定一致。

下面按真实案例、基线、手写机制、结果表和失败修复组织回答；所有数据都是可复现的教学实验。


## 真实案例

场景是客服与账户安全系统中的六条脱敏离线事件。字段包含工单文本、有效 token 数和风险标签；它们模拟真实的数据结构，但样本极小，只用于观察公式和状态变化。


In [1]:
import math  # 导入数学函数以实现训练与掩码公式。
import warnings  # 导入警告控制模块保持输出干净。
warnings.filterwarnings('ignore', message='The pynvml package is deprecated')  # 屏蔽环境依赖的非教学弃用提示。
import torch  # 导入张量计算和自动微分能力。
import torch.nn as nn  # 导入模块基类以手写网络结构。
torch.manual_seed(29)  # 固定随机种子使教学输出可复现。
torch.set_num_threads(1)  # 限制小实验 CPU 线程数。
samples = [  # 构造六条脱敏客服对话作为真实语义样本。
    {'id': 'C01', 'text': '支付重复扣款，申请退款', 'tokens': 6, 'risk': 1},  # 资金风险工单。
    {'id': 'C02', 'text': '收不到登录验证码', 'tokens': 2, 'risk': 0},  # 登录支持工单。
    {'id': 'C03', 'text': '账户有陌生转账记录', 'tokens': 5, 'risk': 1},  # 账户安全工单。
    {'id': 'C04', 'text': '修改订单收货地址', 'tokens': 3, 'risk': 0},  # 售后咨询工单。
    {'id': 'C05', 'text': '银行卡盗刷需要冻结', 'tokens': 7, 'risk': 1},  # 高优先级安全工单。
    {'id': 'C06', 'text': '更正发票抬头信息', 'tokens': 4, 'risk': 0},  # 账单服务工单。
]  # 结束教学数据定义。
features = torch.tensor([[1.0, 0.0, 1.0], [0.0, 1.0, 0.0], [1.0, 0.0, 0.0], [0.0, 0.0, 1.0], [1.0, 1.0, 0.0], [0.0, 1.0, 1.0]])  # 构造三维可解释特征。
labels = torch.tensor([1, 0, 1, 0, 1, 0])  # 构造风险分类标签。
print('教学实验：六条脱敏离线客服事件，只验证机制，不代表线上收益。')  # 声明数据边界。
for row in samples:  # 逐条展示真实语义输入。
    print(f"{row['id']} | token={row['tokens']} | risk={row['risk']} | {row['text']}")  # 输出样本字段。
print(f'特征形状={tuple(features.shape)}，标签={labels.tolist()}')  # 输出张量形状。


教学实验：六条脱敏离线客服事件，只验证机制，不代表线上收益。
C01 | token=6 | risk=1 | 支付重复扣款，申请退款
C02 | token=2 | risk=0 | 收不到登录验证码
C03 | token=5 | risk=1 | 账户有陌生转账记录
C04 | token=3 | risk=0 | 修改订单收货地址
C05 | token=7 | risk=1 | 银行卡盗刷需要冻结
C06 | token=4 | risk=0 | 更正发票抬头信息
特征形状=(6, 3)，标签=[1, 0, 1, 0, 1, 0]


## Baseline / 基线

先在同一批六条事件上运行最简单方案。基线不是稻草人，它提供固定的输入、口径和可比较指标。


In [2]:
documents = [[11, 12, 13, 14], [21, 22], [31, 32, 33], [41, 42, 43, 44, 45], [51], [61, 62, 63]]  # 定义六段不同客服对话的 token id 序列。
padded_slots = sum(max(len(documents[index]), len(documents[index + 1])) * 2 for index in range(0, 6, 2))  # 按两条一 batch 估算普通 padding 槽位。
real_tokens = sum(len(document) for document in documents)  # 统计真实 token 数。
baseline_metric = real_tokens / padded_slots  # 计算普通 padding 的 token 利用率。
print(f'普通两条一 batch：真实 token={real_tokens}，padding 槽位={padded_slots}，利用率={baseline_metric:.2%}')  # 展示未 packing 的浪费。


普通两条一 batch：真实 token=18，padding 槽位=24，利用率=75.00%


## 手写核心实现与中间量

代码保留关键分子分母、mask、梯度、参数组或重算路径，而不让 Trainer 或高层框架隐藏面试问题本身。


In [3]:
block_size = 10  # 设置固定 packed block 长度。
packed_blocks = []  # 保存 token、segment id 和每段边界。
current_tokens = []  # 保存当前 block 的 token。
current_segments = []  # 保存当前 block 的 segment id。
segment_id = 0  # 初始化文档编号。
for document in documents:  # 逐段处理客服对话。
    if len(current_tokens) + len(document) > block_size:  # 当前 block 放不下整段对话时。
        packed_blocks.append({'tokens': current_tokens, 'segments': current_segments})  # 关闭当前 block。
        current_tokens = []  # 开启新 block 的 token 容器。
        current_segments = []  # 开启新 block 的 segment 容器。
    current_tokens.extend(document)  # 将整段 token 拼入当前 block。
    current_segments.extend([segment_id] * len(document))  # 为每个 token 写入对应文档 id。
    segment_id += 1  # 进入下一段文档 id。
packed_blocks.append({'tokens': current_tokens, 'segments': current_segments})  # 保存最后一个非空 block。
packed_slots = len(packed_blocks) * block_size  # 以固定 block 容量估算占用槽位。
core_metric = real_tokens / packed_slots  # 计算 packing 后利用率。
print(f'Packing blocks={packed_blocks}，利用率={core_metric:.2%}')  # 输出 token 与 segment 的中间结构。


Packing blocks=[{'tokens': [11, 12, 13, 14, 21, 22, 31, 32, 33], 'segments': [0, 0, 0, 0, 1, 1, 2, 2, 2]}, {'tokens': [41, 42, 43, 44, 45, 51, 61, 62, 63], 'segments': [3, 3, 3, 3, 3, 4, 5, 5, 5]}]，利用率=90.00%


In [4]:
comparison_rows = [('Baseline', float(baseline_metric)), ('核心机制', float(core_metric))]  # 建立同一口径的结果表。
for name, metric in comparison_rows:  # 逐行输出结果。
    print(f'{name:<8} | 指标={metric:.6f}')  # 展示可读数值对照。


Baseline | 指标=0.750000
核心机制     | 指标=0.900000


## 结果解读

基线和核心输出只在本受控案例中比较。生产实现要处理 EOS、截断、FIM、多轮 label mask 和跨 rank 数据重排；错误 mask 会造成训练泄漏且很难从总 loss 发现。 观察结果时应关注中间量是否符合公式，而不是把六条样本上的数字宣传为线上收益。

## 失败案例

下一个单元故意破坏关键假设，并用实现修复证明该假设为何必要。


In [5]:
first_block = packed_blocks[0]  # 取第一个 packed block 构造 attention mask。
causal_only = [[int(column <= row) for column in range(len(first_block['tokens']))] for row in range(len(first_block['tokens']))]  # 故意只构造因果 mask。
failure_metric = sum(causal_only[4][column] for column in range(4))  # 统计第 5 个 token 能看到多少前一段 token。
segment_mask = [[int(column <= row and first_block['segments'][column] == first_block['segments'][row]) for column in range(len(first_block['tokens']))] for row in range(len(first_block['tokens']))]  # 同时加入因果和文档边界条件。
fix_metric = sum(segment_mask[4][column] for column in range(4))  # 统计修复后第 5 个 token 看到的前一段 token 数。
print(f'失败：仅 causal 时跨文档可见 token={failure_metric}；修复：segment mask 后跨文档可见 token={fix_metric}')  # 展示 concat 不等于样本独立。


失败：仅 causal 时跨文档可见 token=4；修复：segment mask 后跨文档可见 token=0


## 工程取舍、常见坑与延伸追问

**工程取舍：** 生产实现要处理 EOS、截断、FIM、多轮 label mask 和跨 rank 数据重排；错误 mask 会造成训练泄漏且很难从总 loss 发现。

**常见坑：** 只生成 packed token 不生成 segment id，或对每段重置 position 却没有在推理/rope scaling 中保持一致。

**延伸追问：** packing 与 FlashAttention 的 varlen 接口如何对应？为什么 SFT prompt/completion mask 在 packing 后更容易错位？

## 生产差距

本 Notebook 在 CPU/FP32 下处理 6 条离线事件，省略了真实 token packing、分布式同步、混合精度、checkpoint、隐私治理、监控告警和灰度回滚。生产版本必须替换为受审计的数据管道与系统级指标。


In [6]:
assert len(packed_blocks) == 2  # 验证六段文档被拼入两个 block。
assert core_metric > baseline_metric  # 验证 packing 提高了本数据的 token 利用率。
assert failure_metric > fix_metric  # 验证文档边界 mask 阻断跨样本可见性。
assert all(len(block['tokens']) <= block_size for block in packed_blocks)  # 验证没有 block 超出容量。
